# Korean MCQ Benchmark Evaluation

Run a single Korean MCQ benchmark (e.g. KMMLU, CLIcK, HAE-RAE) through EvalHub SDK with MLflow tracking.
For multi-benchmark or unified (accuracy + performance) evaluations, see **3_eval_hub_unified_benchmark/**.


This notebook runs LLM evaluations through the [EvalHub](https://github.com/eval-hub/eval-hub) REST API using the [eval-hub-sdk](https://github.com/eval-hub/eval-hub-sdk) Python client. All results are automatically tracked in **MLflow**.

## EvalHub Features (GA in RHOAI 3.5)

| Feature | Description |
|---------|-------------|
| Interface | Python SDK / REST API |
| Frameworks | lm-evaluation-harness, GuideLLM, RAGAS, LightEval, and more |
| Multi-benchmark | Multiple benchmarks per request |
| Experiment tracking | **Built-in MLflow** (metrics, params, artifacts) |
| Result management | Centralized API + MLflow UI |
| Job management | SDK `client.jobs.cancel()` |

## Prerequisites

- **0_setup/2_eval_hub_setup.ipynb** completed (EvalHub SDK configured)
- `.env` file configured with `MODEL_ENDPOINT`, `MODEL_API_KEY`, `EVALHUB_URL`

## Step 1: Configuration

In [11]:
import os, subprocess
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
MODEL_NAME = os.getenv("MODEL_NAME", "glm-53-flash")
MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", "")
MODEL_API_KEY = os.getenv("MODEL_API_KEY", "")
import sys; sys.path.insert(0, '..')
from utils.port_forward import resolve_evalhub_url
EVALHUB_URL = resolve_evalhub_url(namespace=NAMESPACE)
import subprocess
EVALHUB_AUTH_TOKEN = os.getenv("EVALHUB_AUTH_TOKEN", "")
if not EVALHUB_AUTH_TOKEN:
    _r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    if _r.returncode == 0:
        EVALHUB_AUTH_TOKEN = _r.stdout.strip()
        print("Auth: using oc token")
if not EVALHUB_AUTH_TOKEN:
    _r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
LIMIT = int(os.getenv("LIMIT", "5"))

print(f"Namespace:       {NAMESPACE}")
print(f"Model Name:      {MODEL_NAME}")
print(f"Model Endpoint:  {MODEL_ENDPOINT}")
print(f"EvalHub URL:     {EVALHUB_URL}")
print(f"MLflow URI:      {MLFLOW_TRACKING_URI}")
print(f"Sample Limit:    {LIMIT}")

Using external EvalHub URL: https://evalhub-demo.apps.openshift-cluster.sandbox3031.opentlc.com
Namespace:       demo
Model Name:      glm-53-flash
Model Endpoint:  https://maas.apps.ocp.cloud.rhai-tmm.dev/prelude-maas/glm-53-flash
EvalHub URL:     https://evalhub-demo.apps.openshift-cluster.sandbox3031.opentlc.com
MLflow URI:      https://mlflow.redhat-ods-applications.svc:8443
Sample Limit:    5


## Step 2: Initialize the EvalHub Client

In [12]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
    ExperimentTag,
    JobStatus,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

from utils.model_auth import build_model_config

model = build_model_config(MODEL_ENDPOINT, MODEL_NAME, MODEL_API_KEY, NAMESPACE)

print(f"EvalHub client connected: {EVALHUB_URL}")
print(f"Model target:             {model.name} @ {model.url}")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


Model auth secret 'model-api-key' ready in namespace 'demo'
EvalHub client connected: https://evalhub-demo.apps.openshift-cluster.sandbox3031.opentlc.com
Model target:             glm-53-flash @ https://maas.apps.ocp.cloud.rhai-tmm.dev/prelude-maas/glm-53-flash


## Step 3: Discover Available Korean Benchmarks

Query EvalHub for benchmarks available through the `lm_evaluation_harness` provider and filter for Korean tasks.

In [13]:
KOREAN_PROVIDER_NAME = "Korean MCQ Evaluation"
KOREAN_PROVIDER_ID = None

for p in client.providers.list():
    if p.name == KOREAN_PROVIDER_NAME:
        KOREAN_PROVIDER_ID = p.resource.id
        print(f"Korean MCQ provider found (id={KOREAN_PROVIDER_ID})")
        break

if not KOREAN_PROVIDER_ID:
    print("ERROR: Korean MCQ provider is not registered.")
    print("  → Cluster owner must run 0_setup/2_eval_hub_setup.ipynb Step A-7 or Part B first.")

all_benchmarks = client.benchmarks.list()

korean_keywords = ["kmmlu", "kobest", "haerae", "klue", "korean", "ko_", "click", "hrm8k"]
korean_benchmarks = [
    bm for bm in all_benchmarks
    if any(kw in bm.id.lower() for kw in korean_keywords)
]

print(f"Total benchmarks available: {len(all_benchmarks)}")
print(f"Korean benchmarks found:    {len(korean_benchmarks)}")
print("=" * 70)
for bm in korean_benchmarks:
    metrics_str = ", ".join(bm.metrics[:3]) if bm.metrics else "N/A"
    print(f"  {bm.id:40s}  metrics=[{metrics_str}]")

Korean MCQ provider found (id=9358101f-95cf-4c98-bdf2-b255e6c5b78d)
Total benchmarks available: 218
Korean benchmarks found:    5
  click                                     metrics=[overall_accuracy, category_accuracy, supercategory_accuracy]
  haerae                                    metrics=[overall_accuracy, category_accuracy]
  kmmlu                                     metrics=[overall_accuracy, category_accuracy, supercategory_accuracy]
  kmmlu_hard                                metrics=[overall_accuracy, category_accuracy, supercategory_accuracy]
  kobest_boolq                              metrics=[overall_accuracy]


---

## Evaluation 1: Single Korean Benchmark

Start with a single benchmark to verify the pipeline works end-to-end.

### 1-A: Submit the Job

In [14]:
single_request = JobSubmissionRequest(
    name="haerae-eval",
    description="HAE-RAE benchmark - Korean language understanding",
    tags=["korean", "haerae", MODEL_NAME],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="haerae",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 1024, "limit": LIMIT},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-haerae-eval",
        tags=[
            ExperimentTag(key="language", value="korean"),
            ExperimentTag(key="benchmark_suite", value="haerae"),
            ExperimentTag(key="evaluation_type", value="full"),
        ],
    ),
)

job = client.jobs.submit(single_request)

print(f"Job submitted successfully!")
print(f"  Job ID:        {job.id}")
print(f"  Name:          {job.name}")
print(f"  State:         {job.state.value}")
print(f"  MLflow Exp ID: {job.resource.mlflow_experiment_id or 'pending'}")

Job submitted successfully!
  Job ID:        096adf59-e751-4e5b-bd3d-309495c01fd2
  Name:          kmmlu-eval
  State:         pending
  MLflow Exp ID: 28


### 1-B: Monitor Progress

In [15]:
import time
from datetime import datetime

TERMINAL_STATES = {
    JobStatus.COMPLETED,
    JobStatus.FAILED,
    JobStatus.CANCELLED,
    JobStatus.PARTIALLY_FAILED,
}


def wait_for_job(client, job_id, poll_interval=10, max_wait=600):
    """Poll job status until terminal state or timeout."""
    start = time.time()
    print(f"Monitoring job {job_id}...")
    print("-" * 70)

    while time.time() - start < max_wait:
        status = client.jobs.get(job_id)
        state = status.effective_state
        elapsed = int(time.time() - start)

        msg = ""
        if status.status and status.status.message:
            msg = f" | {status.status.message.message}"

        bm_info = ""
        if status.status and status.status.benchmarks:
            bm_states = [f"{b.id}={b.state.value}" for b in status.status.benchmarks]
            bm_info = f" | benchmarks: {', '.join(bm_states)}"

        print(f"  [{elapsed:>4d}s] {state.value:>16s}{msg}{bm_info}")

        if state in TERMINAL_STATES:
            break

        time.sleep(poll_interval)

    print("-" * 70)
    print(f"Final state: {state.value} (elapsed: {elapsed}s)")
    return status


completed_job = wait_for_job(client, job.id)

Monitoring job 096adf59-e751-4e5b-bd3d-309495c01fd2...
----------------------------------------------------------------------
  [   0s]          pending | Evaluation job created
  [  11s]          running | Evaluation job is running | benchmarks: kmmlu=running
  [  23s]          running | Evaluation job is running | benchmarks: kmmlu=running
  [  34s]        completed | Evaluation job is completed | benchmarks: kmmlu=completed
----------------------------------------------------------------------
Final state: completed (elapsed: 34s)


### 1-C: View Results

In [16]:
def display_job_results(job):
    """Display evaluation results with MLflow links."""
    if not job.results:
        print("No results available.")
        return

    print("Evaluation Results")
    print("=" * 70)

    if job.results.mlflow_experiment_url:
        print(f"\n  MLflow Experiment: {job.results.mlflow_experiment_url}")

    for bm in job.results.benchmarks:
        print(f"\n  Benchmark: {bm.id}")
        print(f"  Provider:  {bm.provider_id}")
        if bm.mlflow_run_id:
            print(f"  MLflow Run: {bm.mlflow_run_id}")

        if bm.metrics:
            print(f"  Metrics:")
            for name, value in bm.metrics.items():
                if isinstance(value, float):
                    print(f"    {name:30s} = {value:.4f}")
                else:
                    print(f"    {name:30s} = {value}")
        else:
            print("  Metrics: (none)")


display_job_results(completed_job)

Evaluation Results

  MLflow Experiment: https://mlflow.redhat-ods-applications.svc:8443/mlflow/api/2.0/mlflow/experiments

  Benchmark: kmmlu
  Provider:  9358101f-95cf-4c98-bdf2-b255e6c5b78d
  MLflow Run: fccf8412ea614e5390c319e034a97c64
  Metrics:
    category_accuracy.Accounting   = 40
    overall_accuracy               = 40
    supercategory_accuracy.HUMSS   = 40


---

## Step 4: Job Management

List, inspect, and manage all evaluation jobs.

### List All Jobs

In [17]:
jobs_list = client.jobs.list()

print(f"Total Jobs: {len(jobs_list)}")
print("=" * 90)
print(f"{'State':>16s}  {'Job ID':12s}  {'Name':30s}  {'Experiment':25s}  Benchmarks")
print("-" * 90)
for j in jobs_list:
    state = j.effective_state.value
    exp = j.experiment.name if j.experiment else "N/A"
    bms = [b.id for b in j.benchmarks] if j.benchmarks else []
    print(f"{state:>16s}  {j.id[:12]:12s}  {j.name[:30]:30s}  {exp[:25]:25s}  {bms}")

Total Jobs: 3
           State  Job ID        Name                            Experiment                 Benchmarks
------------------------------------------------------------------------------------------
       completed  e92360ed-f38  kmmlu-auth-v2                   kmmlu-auth-v2              ['kmmlu']
       completed  29fa5c38-7a2  guidellm-constant-test          guidellm-constant-final    ['constant']
       completed  096adf59-e75  kmmlu-eval                      korean-kmmlu-eval          ['kmmlu']


### Inspect a Specific Job

Replace `JOB_ID` with the job you want to inspect.

In [18]:
# Replace with actual job ID:
# JOB_ID = "your-job-id-here"
# inspected = client.jobs.get(JOB_ID)
# display_job_results(inspected)

---

## Step 5: Export Results

### Export to Markdown

In [19]:
def collect_results_table(client):
    """Collect benchmark metrics from completed jobs into a comparison dict."""
    jobs_list = client.jobs.list()
    table = {}
    for j in jobs_list:
        if j.effective_state != JobStatus.COMPLETED or not j.results:
            continue
        for bm in j.results.benchmarks:
            if bm.id not in table:
                table[bm.id] = {}
            table[bm.id][j.name] = bm.metrics
    return table


comparison = collect_results_table(client)


def results_to_markdown(comparison, title="EvalHub Benchmark Results"):
    """Convert comparison results to markdown table."""
    if not comparison:
        return "No results available."

    lines = [f"## {title}", ""]

    for benchmark_id, job_results in comparison.items():
        lines.append(f"### {benchmark_id}")
        lines.append("")

        all_metrics = set()
        for metrics in job_results.values():
            all_metrics.update(metrics.keys())
        all_metrics = sorted(all_metrics)

        job_names = sorted(job_results.keys())
        header = "| Metric | " + " | ".join(job_names) + " |"
        sep = "|---" + "|---" * len(job_names) + "|"
        lines.extend([header, sep])

        for metric in all_metrics:
            row = f"| {metric} |"
            for job_name in job_names:
                val = job_results.get(job_name, {}).get(metric, "-")
                if isinstance(val, float):
                    row += f" {val:.4f} |"
                else:
                    row += f" {val} |"
            lines.append(row)

        lines.append("")

    return "\n".join(lines)


md = results_to_markdown(comparison)
print(md)

## EvalHub Benchmark Results

### kmmlu

| Metric | kmmlu-auth-v2 | kmmlu-eval |
|---|---|---|
| category_accuracy.Accounting | 40 | 40 |
| overall_accuracy | 40 | 40 |
| supercategory_accuracy.HUMSS | 40 | 40 |

### constant

| Metric | guidellm-constant-test |
|---|---|
| mean_itl_ms | 0 |
| mean_ttft_ms | 0 |
| output_tokens_per_second | 32.5370 |
| prompt_tokens_per_second | 74.8350 |
| requests_per_second | 1.0000 |



### Save Results to JSON

In [20]:
import json
from pathlib import Path

results_root = Path("../results")

jobs_list = client.jobs.list()
saved = 0
for j in jobs_list:
    if j.effective_state != JobStatus.COMPLETED or not j.results:
        continue

    model_name = j.model.name if j.model else "unknown"
    model_dir = results_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    job_data = {
        "job_id": j.id,
        "name": j.name,
        "model": {"url": j.model.url, "name": j.model.name},
        "experiment": j.experiment.name if j.experiment else None,
        "benchmarks": [
            {
                "id": bm.id,
                "provider_id": bm.provider_id,
                "metrics": bm.metrics,
                "mlflow_run_id": bm.mlflow_run_id,
            }
            for bm in j.results.benchmarks
        ],
    }

    filename = f"{j.name}_{j.id[:8]}.json"
    output_path = model_dir / filename
    with open(output_path, "w") as f:
        json.dump(job_data, f, indent=2, default=str)
    print(f"Saved: {output_path}")
    saved += 1

print(f"\n{saved} result(s) saved to {results_root.resolve()}/<model-name>/")

Saved: ../results/glm-53-flash/kmmlu-auth-v2_e92360ed.json
Saved: ../results/glm-53-flash/guidellm-constant-test_29fa5c38.json
Saved: ../results/glm-53-flash/kmmlu-eval_096adf59.json

3 result(s) saved to /Users/hyochoi/dev/rhoai-evalhub-lab/results/<model-name>/


---

## Summary

This notebook demonstrated a single Korean MCQ benchmark evaluation:

1. **Configuration** and EvalHub client setup
2. **Single benchmark** evaluation (KMMLU) with MLflow tracking
3. **Job management** -- list and inspect jobs
4. **Export** -- Markdown and JSON output

### Next Steps

- **2_summarize_results.ipynb** -- Aggregate results and generate HTML/Markdown reports
- **3_eval_hub_unified_benchmark/1_unified_benchmark.ipynb** -- Run multi-benchmark + GuideLLM unified evaluation
- **1_eval_hub_guidellm_benchmark/1_guidellm_benchmark.ipynb** -- Standalone GuideLLM performance profiling
